# Priprava podatkov

## Nalaganje podatkov

In [1]:
import pandas as pd
from pathlib import Path

Zberemo vse tekme za specifičen razpon sezon v en DataFrame

In [2]:
folder = Path("./data")

dfs = []

for file in folder.glob("atp_matches_2[0-9][0-9][0-9].csv"):
    year = int(file.stem.split("_")[-1])
    
    if 2000 <= year <= 2026:
        dfs.append(pd.read_csv(file).assign(source_year=year))
        
df = pd.concat(dfs, ignore_index=True)

Popravimo format datumov turnirjev

In [3]:
df["tourney_date"] = pd.to_datetime(df["tourney_date"].astype(str), format="%Y%m%d", errors="coerce")
df = df.dropna(subset=["tourney_date"])
df["year"] = df["tourney_date"].dt.year

## Čiščenje

Odstranimo tekme ki nimajo obveznih podatkov, niso bile normalno zaključene (predaje in walkoverji) ali pa so del Davis Cup-a.

In [4]:
obvezni_stolpci = [
  'tourney_date',
  'surface',
  'best_of',
  'round',
  'winner_id',
  'loser_id',
  'winner_rank',
  'loser_rank',
  'winner_rank_points',
  'loser_rank_points'
]
df = df.dropna(subset=obvezni_stolpci)

df = df[~df["score"].astype(str).str.contains("W/O|Walkover", case=False, na=False)]
df = df[~df["score"].astype(str).str.contains("RET|Retired", case=False, na=False)]
df = df[~df["score"].astype(str).str.contains("Def|Default", case=False, na=False)]

df = df[df["tourney_level"].isin(["G", "M", "A", "F"])]


Izbrišemo stolpce ki so znani šele po koncu tekme in bi povzročali data leakage

In [5]:
data_leakage_columns = [
    "score",
    "minutes",
    "w_ace",
    "w_df",
    "w_svpt",
    "w_1stIn",
    "w_1stWon",
    "w_2ndWon",
    "w_SvGms",
    "w_bpSaved",
    "w_bpFaced",
    "l_ace",
    "l_df",
    "l_svpt",
    "l_1stIn",
    "l_1stWon",
    "l_2ndWon",
    "l_SvGms",
    "l_bpSaved",
    "l_bpFaced",
]

df = df.drop(columns=data_leakage_columns)

Uporabili smo samo ATP main draw tekme iz let 2000–2026. Izločili smo tekme brez datuma, površine, identifikacije igralcev ali rankinga obeh igralcev. Izločili smo tudi walkoverje, predaje in default tekme, saj ne predstavljajo standardno odigranih dvobojev. V model nismo vključili statistik, ki nastanejo med tekmo, kot so asi, dvojne napake, trajanje tekme in podatki o servisnih točkah, ker te informacije pred začetkom tekme niso znane.

## Shranjevanje

Preostale tekme shranimo v ./data/processed/matches_clean.csv

In [6]:
processed_dir = Path("./data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(processed_dir / "matches_clean.csv", index=False)